In [25]:
from pathlib import Path
import sys

import numpy as np

notebook_dir = Path.cwd()
if not (notebook_dir / 'valid_predicted_shapes.npz').exists():
    candidate_dir = Path.cwd() / 'centerline_tape'
    if (candidate_dir / 'valid_predicted_shapes.npz').exists():
        notebook_dir = candidate_dir

sys.path.append(str(notebook_dir))
from logging_data import export_rod_shell_data, logDataForRendering

In [26]:
# code to extract centerline for rendering

source_of_data = 0  # 1 for pydismech, 0 for dismech-jax
batch_index = 3      # used only when qs has shape (n_batches, n_time_steps, n_dofs)

case = 1
# 0: use ground truth data
# 1: use predicted data my model
# 2: use predicted data from baseline

train_or_test = 'test'  # set to 'train' if you want to use training data

# read npz file to get qs
if case==0:
    if train_or_test == 'train':
        data = np.load(notebook_dir / 'n7_tube_train_dataset.npz')
    else:
        data = np.load(notebook_dir / 'n7_tube_test_dataset.npz')
    qs = data['qs']
    rod_file = notebook_dir / 'rawDataRod_gd_centerline.txt'
    rod_js = notebook_dir / 'rodData_gd_centerline.js'
elif case==1:
    if train_or_test == 'train':
        data = np.load(notebook_dir / 'train_predicted_shapes.npz')
        
    else:
        data = np.load(notebook_dir / 'valid_predicted_shapes.npz')
    qs = data['qs_pred']
    rod_file = notebook_dir / 'rawDataRod_centerline.txt'
    rod_js = notebook_dir / 'rodData_centerline.js'

elif case==2:
    if train_or_test == 'train':
        data = np.load(notebook_dir / 'train_predicted_shapes_diag_baseline.npz')
    else:
        data = np.load(notebook_dir / 'valid_predicted_shapes_diag_baseline.npz')
    qs = data['qs_pred']
    rod_file = notebook_dir / 'rawDataRod_centerline_diag_baseline.txt'
    rod_js = notebook_dir / 'rodData_centerline_diag_baseline.js'


# if qs is 3 dimensional: (n_batches, n_time_steps, n_dofs)
# if qs is 2 dimensional: (n_time_steps, n_dofs)
if qs.ndim == 3:
    qs_for_rendering = qs[batch_index]
elif qs.ndim == 2:
    qs_for_rendering = qs
else:
    raise ValueError(f'Expected qs to be 2D or 3D, got shape {qs.shape}')

n_time_steps, n_dofs = qs_for_rendering.shape

# infer no. of nodes
# pydismech: 3*n_nodes nodal dofs + (n_nodes - 1) edge thetas = 4*n_nodes - 1
# dismech-jax: same dof count, but theta values are interleaved after each edge node
n_nodes = (n_dofs + 1) // 4
if n_dofs != 4 * n_nodes - 1:
    raise ValueError(f'Could not infer node count from {n_dofs} dofs')

# the data has both nodes and edge thetas so only get the nodes
# if the data is gotten using pydismech: it has the format [node1, node2, node3, node4, edge1, edge2, edge3]
# if the data is from training from dismech-jax: it has the format [node1, edge1, node2, edge2, node3, edge3, node4]
if source_of_data == 1:
    # for pydismech, get nodes : 3 dofs for each node until 3*n_nodes
    centerline = qs_for_rendering[:, :3 * n_nodes]
else:
    # for dismech-jax, get nodes : 3 dofs for each node followed by 1 dof for each edge
    # so remove the edge thetas: edge thetas are at indices 3, 7, 11, ...
    theta_indices = np.arange(3, n_dofs, 4)
    centerline = np.delete(qs_for_rendering, theta_indices, axis=1)

if centerline.shape != (n_time_steps, 3 * n_nodes):
    raise ValueError(
        f'Expected centerline shape {(n_time_steps, 3 * n_nodes)}, got {centerline.shape}'
    )

time = np.arange(n_time_steps).reshape(-1, 1)

# use logDataForRendering() to get the .txt file
logDataForRendering(centerline, time, n_nodes, n_time_steps, rod_file=str(rod_file))

# use export_rod_shell_data(__.txt) to get the .js file used for rendering
export_rod_shell_data(
    n_nodes,
    rod_file=str(rod_file),
    rod_js=str(rod_js),
    rod_radius=0.1,
    scaleFactor=10,
)

print(f'qs shape: {qs.shape}')
print(f'centerline shape: {centerline.shape}')
print(f'n_nodes: {n_nodes}, n_time_steps: {n_time_steps}')
print(f'wrote {rod_file}')
print(f'wrote {rod_js}')


(29, 22) (29, 1) (29, 21)
qs shape: (6, 29, 27)
centerline shape: (29, 21)
n_nodes: 7, n_time_steps: 29
wrote /Users/radha/Desktop/Threejs_rendering/centerline_tube/rawDataRod_centerline.txt
wrote /Users/radha/Desktop/Threejs_rendering/centerline_tube/rodData_centerline.js
